# Stage 1 — Build Full Dataset
**[STUDENT VERSION — fill in the blanks]**

Stage 1 là bước “gom dữ liệu” để tạo một bảng điểm đến hoàn chỉnh. Ở đây, chúng ta hợp nhất 3 nguồn thông tin khác nhau:
- điểm ngữ nghĩa từ Stage 0,
- thuộc tính thực tế như budget / family / access / crowd,
- và tọa độ địa lý lat/lng.

Sau khi ghép, mỗi điểm đến sẽ có một vector đặc trưng thống nhất. Đây là đầu vào trực tiếp cho Stage 2, nơi hệ thống sẽ so khớp nhu cầu người dùng với điểm đến bằng cosine similarity.

- Input: `stage0_tfidf_scores.csv` — 8 semantic scores từ Stage 0
- Input: `stage1_factual.csv` — budget, family, access, crowd, best_months
- Input: `stage1_coordinates.csv` — lat, lng
- Output: `stage1_dataset.csv` và `stage1_dataset.json`

> 💡 Cells marked `# TODO` require you to fill in the code.

## Bức tranh toàn pipeline

```text
Stage 0: mô tả địa điểm ──TF-IDF──> 8 điểm ngữ nghĩa
                                      │
Stage 1: factual + tọa độ ────────────┼──> feature table hoàn chỉnh
                                      │
Stage 2: sở thích + tháng đi ─────────┘──> xếp hạng điểm đến
                                                │
Stage 3: số ngày + lat/lng ─────────────────────┘──> chia ngày và xếp tuyến
```

Stage 1 chưa huấn luyện mô hình. Đây là bước **ETL và chuẩn bị đặc trưng**: đọc dữ liệu từ nhiều nguồn, kiểm tra chúng có cùng nói về một địa điểm, đổi về đúng kiểu dữ liệu rồi ghi ra một bảng thống nhất cho các stage sau.

### Mục tiêu học tập

Sau stage này, bạn có thể:

1. mô tả được dữ liệu đi từ nguồn nào đến cột nào trong dataset cuối;
2. phân biệt khóa định danh, feature dùng để gợi ý, dữ liệu ngữ cảnh và tọa độ;
3. giải thích vì sao phải join theo `place` thay vì theo số thứ tự dòng;
4. phát hiện dữ liệu thiếu hoặc sai kiểu trước khi lỗi lan sang recommender;
5. tạo cùng một dataset ở định dạng CSV và JSON cho các mục đích sử dụng khác nhau.

In [61]:
import csv, json
from pathlib import Path
import pandas as pd
import math

In [62]:
BASE_DIR     = Path('.')
TFIDF_CSV    = BASE_DIR / 'stage0_tfidf_scores_rerun.csv'
FACTUAL_CSV  = BASE_DIR / '../input/stage1/stage1_factual.csv'
COORDS_CSV   = BASE_DIR / '../input/stage1/stage1_coordinates.csv'
DATASET_CSV  = BASE_DIR / '../input/stage1/stage1_dataset.csv'
DATASET_JSON = BASE_DIR / '../input/stage1/stage1_dataset.json'

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

tfidf_rows   = read_csv(TFIDF_CSV)
factual_rows = read_csv(FACTUAL_CSV)
coord_rows   = read_csv(COORDS_CSV)

def min_max_normalization(row, reverse = False):
    if reverse:
        row = [x * -1 for x in row]
    mn = min(row)
    mx = max(row)
    denom = mx - mn
    res = []
    for value in row:
        res.append((value - mn) / denom if denom > 0 else 0)
    return res

def logarithm_normalization(row, reverse = False):
    mn = math.log(min(row) + 1)
    mx = math.log(max(row) + 1)
    denom = mx - mn
    res = []
    for value in row:
        res.append(((math.log(value + 1) - mn) / denom if reverse == False else (mx - math.log(value + 1)) / denom) if denom > 0 else 0)
    return res

def normalize_data(data, features):
    n = len(features)
    features = [feature.strip() for feature in features]
    cands = [[] for _ in range(n)]
    for item in data:
        cursor = 0
        for key, value in item.items():
            key = key.strip()
            flag = False
            for feature in features:
                if (key.strip() == feature.strip()):
                    flag = True
                    break
            if (flag):
                cands[cursor].append(float(value))
                cursor += 1
    for i in range(len(cands)):
        if (features[i] == "budget"):
            cands[i] = logarithm_normalization(cands[i], True)
        else:
            cands[i] = min_max_normalization(cands[i])
    index = 0
    for item in data:
        for i in range(n):
            feature = features[i]
            if feature in item:
                item[feature] = cands[i][index]
        index += 1

normalize_data(factual_rows, ["budget", "family", "crowd"])

print(f'TF-IDF rows: {len(tfidf_rows)}')
print(f'Factual rows: {len(factual_rows)}')
print(f'Coord rows: {len(coord_rows)}')


TF-IDF rows: 42
Factual rows: 42
Coord rows: 42


## ETL, feature table và data contract

**ETL** là viết tắt của Extract – Transform – Load:

- **Extract:** đọc ba file CSV thành các dòng dữ liệu;
- **Transform:** lập chỉ mục, kiểm tra khóa, đổi kiểu và ghép các cột;
- **Load:** ghi feature table hoàn chỉnh ra CSV và JSON.

Có thể hình dung `stage1_dataset` như một **feature table thu nhỏ**: mỗi dòng là một điểm đến, mỗi cột có tên, kiểu và ý nghĩa cố định. Tập quy tắc đó gọi là **data contract**. Các stage sau chỉ hoạt động đúng khi Stage 1 giữ đúng contract này.

| Nhóm | Cột | Kiểu/miền giá trị | Vai trò và ý nghĩa |
|---|---|---|---|
| Định danh | `place`, `province` | chuỗi | `place` là khóa nối ba nguồn; `province` là thông tin mô tả |
| Ngữ nghĩa | `beach` … `photo` | float trong `[0, 1]` | mức liên quan đến 8 chủ đề từ Stage 0 |
| Thực tế | `budget` | float trong `[0, 1]` | càng cao càng **rẻ/tiết kiệm, thân thiện với ngân sách** |
| Thực tế | `family` | float trong `[0, 1]` | càng cao càng phù hợp gia đình |
| Thực tế | `access` | float trong `[0, 1]` | càng cao càng dễ tiếp cận |
| Thực tế | `crowd` | float trong `[0, 1]` | càng cao càng **đông**, càng thấp càng vắng |
| Không gian | `lat`, `lng` | float | tọa độ chỉ dùng để chia cụm và tính tuyến ở Stage 3, không đi vào vector cosine của Stage 2 |
| Ngữ cảnh | `best_months` | chuỗi | danh sách tháng tốt; sẽ được mã hóa theo tháng người dùng hỏi ở Stage 2 |

> **Feature semantics quan trọng không kém giá trị số.** Ví dụ `crowd = 1` phải luôn mang cùng nghĩa “đông” ở cả dữ liệu điểm đến và vector sở thích. Nếu một phía hiểu là “đông” còn phía kia hiểu là “muốn tránh đông”, cosine similarity sẽ so sánh hai trục ngược nghĩa.

## 🔧 TODO 1 — Index by place name

Convert 3 list of rows thành 3 dict, key là `place`.  
Mục đích: tra cứu nhanh theo tên điểm thay vì loop.

### Vì sao không ghép theo vị trí dòng?

Giả sử file TF-IDF có thứ tự `[Hội An, Huế]`, còn file tọa độ có thứ tự `[Huế, Hội An]`. Ghép dòng 1 với dòng 1 sẽ gắn tọa độ Huế cho Hội An dù cả hai file đều đủ dữ liệu. Thứ tự dòng chỉ là cách lưu; `place` mới là danh tính của bản ghi.

Lập một dictionary theo khóa có hai lợi ích:

- bản ghi được ghép theo **ý nghĩa**, không phụ thuộc thứ tự file;
- sau bước tạo index `O(n)`, mỗi lần tra một `place` có độ phức tạp trung bình `O(1)`. Nếu mỗi địa điểm lại quét toàn bộ danh sách để tìm, toàn bộ phép ghép có thể tăng thành `O(n²)`.

```python
# Ví dụ kết quả mong muốn:
tfidf['Phong Nha'] → {'place': 'Phong Nha', 'beach': '0.072', ...}
```


In [63]:
# TODO 1: Index 3 data sources by place name
tfidf   = {r['place'].lower().strip() : r for r in tfidf_rows}  # ← {r['place']: r for r in tfidf_rows}
factual = {r['place'].lower().strip() : r for r in factual_rows}  # ← tương tự
coords  = {r['place'].lower().strip() : r for r in coord_rows}  # ← tương tự

for key, value in factual.items():
    print(value["budget"])

# Kiểm tra
print(f'Sample tfidf keys: {list(tfidf.keys())[:3]}')


0.32633240045100004
0.0
0.11854882737322628
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.4052202118753022
1.0
1.0
1.0
1.0
1.0
1.0
0.4052202118753022
1.0
1.0
0.32901445247558064
1.0
0.34482865587491585
0.34482865587491585
1.0
1.0
1.0
1.0
1.0
1.0
0.4052202118753022
1.0
0.34482865587491585
0.39079792668628227
1.0
Sample tfidf keys: ['phong nha', 'hang sơn đoòng', 'hang va']


## 🔧 TODO 2 — Validation

Kiểm tra này đảm bảo ba nguồn dữ liệu thật sự khớp nhau theo cùng danh sách `place`.
Nếu thiếu một điểm ở factual hoặc coordinates, kết quả merge sẽ bị lệch cột hoặc mất dữ liệu mà khó phát hiện về sau.

Nói cách khác, đây là bước “chặn lỗi sớm” trước khi tạo dataset cuối cùng.

Kiểm tra: tất cả điểm trong `tfidf` có mặt trong `factual` và `coords` không?  
Nếu thiếu → raise ValueError trước khi merge.

### Những lỗi dữ liệu cần nhận biết

TODO của bài tập thực hiện kiểm tra thiếu bản ghi tối thiểu. Trong một pipeline production, data contract còn nên kiểm tra:

- **trùng khóa:** hai dòng cùng `place` có thể bị dictionary giữ lại một dòng và che mất dòng kia;
- **bản ghi thừa:** factual hoặc coordinates có địa điểm không xuất hiện trong nguồn TF-IDF;
- **khác Unicode/chuỗi:** `Hội An`, `Hoi An` và `Hội An ` là ba khóa khác nhau đối với Python; khoảng trắng, viết hoa và chuẩn Unicode cần được thống nhất từ nguồn;
- **thiếu cột hoặc sai kiểu:** cột số rỗng, chứa chữ hoặc nằm ngoài miền `[0, 1]`;
- **tọa độ không hợp lệ:** latitude phải trong `[-90, 90]`, longitude trong `[-180, 180]`.

> Mục đích của validation không phải “sửa đoán” dữ liệu, mà là dừng sớm với thông báo đủ rõ để sửa đúng nguồn.

In [64]:
# TODO 2: Validation — tìm điểm bị thiếu
missing_factual = [p for p in tfidf if p not in factual]  # ← [p for p in tfidf if p not in factual]
missing_coords  = [p for p in tfidf if p not in coords]  # ← tương tự với coords

if missing_factual:
    raise ValueError(f'Missing factual data for: {missing_factual}')
if missing_coords:
    raise ValueError(f'Missing coordinates for: {missing_coords}')

print('Validation passed.')


Validation passed.


## 🔧 TODO 3 — Merge Dataset

Với mỗi điểm đến, tạo 1 dict gồm:
- `place`, `province`, `lat`, `lng`
- 8 semantic features từ `tfidf` (convert sang float)
- 4 factual features từ `factual` (convert sang float)
- `best_months` từ `factual` — **giữ nguyên dạng string, KHÔNG convert sang float**

### Kiểu dữ liệu và feature tính muộn

`csv.DictReader` trả về chuỗi cho mọi ô, kể cả khi CSV ghi `0.75` hoặc `16.47`. Vì vậy semantic scores, factual scores, `lat` và `lng` phải được ép sang `float` trước khi tính toán. Việc ép kiểu cũng là một phép kiểm tra sớm: dữ liệu như chuỗi rỗng hoặc `unknown` sẽ gây lỗi ngay tại nơi tạo dataset thay vì âm thầm đi tiếp.

> ⚠️ `best_months` là string vì giá trị của nó phụ thuộc vào tháng user nhập.
> Chỉ được encode thành số lúc tính cosine ở Stage 2, không phải ở đây.

Cách giữ dữ liệu thô rồi chỉ mã hóa khi có truy vấn được gọi là **late binding** (tính feature muộn). Ví dụ cùng một `best_months`, điểm số mùa có thể là `1.0` với người đi tháng 6 nhưng là `0.0` với người đi tháng 10. Nếu Stage 1 mã hóa sẵn một con số duy nhất, dataset sẽ không thể trả lời đúng cho cả hai người.

Tương tự, `lat` và `lng` được giữ như metadata không gian. Stage 2 không dùng chúng để đo sở thích; Stage 3 mới dùng chúng để gom các điểm gần nhau và ước lượng quãng đường.

In [65]:
SEMANTIC = ['beach', 'history', 'food', 'nature', 'adventure', 'culture', 'relax', 'photo']
FACTUAL  = ['budget', 'family', 'crowd']

dataset = []
for place in tfidf:
    row = {
        'place':    place,
        'province': tfidf[place]['province'],
        'lat':      float(coords[place]['lat']),  # ← float(coords[place]['lat'])
        'lng':      float(coords[place]['lng']),  # ← float(coords[place]['lng'])
    }
    # TODO: thêm 8 semantic features từ tfidf (convert sang float)
    for f in SEMANTIC:
        row[f] = float(tfidf[place][f])  # ← float(tfidf[place][f])

    # TODO: thêm 4 factual features từ factual (convert sang float)
    for f in FACTUAL:
        row[f] = float(factual[place][f])  # ← float(factual[place][f])

    # TODO: thêm best_months — GIỮ NGUYÊN STRING, không convert
    row['best_months'] = factual[place]['best_months']  # ← factual[place]['best_months']

    dataset.append(row)

print(f'Dataset: {len(dataset)} destinations')
print(f'Sample: {list(dataset[0].keys())}')


Dataset: 42 destinations
Sample: ['place', 'province', 'lat', 'lng', 'beach', 'history', 'food', 'nature', 'adventure', 'culture', 'relax', 'photo', 'budget', 'family', 'crowd', 'best_months']


## Vì sao lưu cả CSV và JSON?

Hai file chứa cùng các bản ghi nhưng phục vụ cách dùng khác nhau:

- **CSV** là bảng phẳng, dễ mở bằng spreadsheet, so sánh cột và kiểm tra dữ liệu bằng mắt;
- **JSON** giữ rõ kiểu số/chuỗi và thuận tiện để chương trình khác hoặc một API trong tương lai đọc.

Việc tạo file JSON chưa biến notebook thành API. Repo hiện tại vẫn là pipeline chạy theo batch: chạy notebook, tạo artifact, rồi notebook kế tiếp đọc artifact đó.

In [66]:
# Save
headers = ['place','province','lat','lng'] + SEMANTIC + FACTUAL + ['best_months']
with DATASET_CSV.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(dataset)
with DATASET_JSON.open('w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)
print(f'Saved: {DATASET_CSV.name}')
print(f'Saved: {DATASET_JSON.name}')


Saved: stage1_dataset.csv
Saved: stage1_dataset.json


In [67]:
# Score range check — tất cả phải trong [0, 1]
print('Feature score ranges:')
for feat in SEMANTIC + FACTUAL:
    vals = [row[feat] for row in dataset]
    print(f'  {feat:<12} min={min(vals):.3f}  max={max(vals):.3f}')


Feature score ranges:
  beach        min=0.000  max=1.000
  history      min=0.000  max=1.000
  food         min=0.000  max=1.000
  nature       min=0.000  max=1.000
  adventure    min=0.000  max=1.000
  culture      min=0.000  max=1.000
  relax        min=0.000  max=1.000
  photo        min=0.000  max=1.000
  budget       min=0.000  max=1.000
  family       min=0.000  max=1.000
  crowd        min=0.000  max=1.000


## ✅ Tự kiểm tra sau khi hoàn thành

Đừng chỉ dựa vào dòng `Saved`. Hãy kiểm tra cả cấu trúc lẫn ý nghĩa dữ liệu:

- [ ] Số dòng output bằng số địa điểm TF-IDF và `place` không bị trùng.
- [ ] Mỗi địa điểm có đủ factual data và tọa độ của **đúng địa điểm đó**.
- [ ] Tất cả semantic/factual features là số thực trong `[0, 1]`.
- [ ] `budget` cao vẫn mang nghĩa rẻ; `crowd` cao vẫn mang nghĩa đông ở mọi stage.
- [ ] `lat`, `lng` là số hợp lệ nhưng không bị đưa vào vector sở thích Stage 2.
- [ ] `best_months` vẫn là chuỗi để Stage 2 tính theo từng truy vấn.
- [ ] CSV và JSON có cùng số bản ghi, cùng bộ cột và vẫn đọc lại được.

Nếu bạn có thể giải thích vì sao ghép theo thứ tự dòng là nguy hiểm và vì sao `best_months` chưa nên đổi thành số ở Stage 1, bạn đã nắm được hai ý chính của bài.